# Plan

local save → push to server when online → pull from server in Streamlit

**Piece 1** — Unity (C#): On save, write the JSON to local storage as you already do. Then attempt to POST it to your Cloud Function endpoint. If it fails (no internet), flag it for retry. On app start or when connectivity changes, retry any unsent files. Mark files as "synced" once the server confirms receipt.

**Piece 2** — Cloud Function: Receives the JSON, validates it minimally (is it valid JSON, does it have a player_id), writes it as a document to a Firestore collection like game_saves/{auto-generated-id}. Adds a uploaded_at server timestamp. Returns success/failure.

**Piece 3** — Streamlit: On your existing data loading page, add an option to "Load from server" alongside the manual upload. Query Firestore for documents, let the user pick which session to visualize (by player_id, date, etc.), load it into the same data structures your app already uses. You could poll automatically or just have a "Refresh" button — for a research tool, a button is simpler and more predictable.

## Workflow
```
Game session ends
       ↓
Save JSON to local folder: Unsent/
       ↓
Try to upload to Firestore
       ↓
  ┌─── Success? ───┐
  ↓                 ↓
 YES               NO
  ↓                 ↓
Move file       File stays
to Sent/        in Unsent/
                    ↓
              Retry later
              (app start, 
              or periodic check)
```

1. Unity to Firestore
UnityWebRequest → Cloud Function → Firestore — Unity just does a plain HTTP POST with the JSON body to a Firebase Cloud Function URL. The Cloud Function (a few lines of Node.js or Python) writes it to Firestore. This is simpler on the Unity side — you already know how to do HTTP requests in C# — and the offline queuing you'd handle yourself by saving to local storage first, then posting when connectivity returns. You're probably already saving locally anyway.





# FIle format

filename = {sessionId}.json

```json
{
  "sessionId": "2026-08-03_14h30m25_P01",
  "playerId": "P01",
  "startTime": "2026-08-03T14:28:02Z",
  "timestamp": "2026-08-03T14:30:25Z",
  "deviceName": "TabletLabo03",
  
  ...rest of your game data...
}
```

# Firestore

Good. Before writing the uploader class, let me explain how talking to Firestore from Unity actually works, because there's a quirk.

The Firestore REST API doesn't accept raw JSON. You can't just POST your save file as-is. Firestore wraps every field in typed objects. So where your JSON has:

```json
{ "playerId": "P01" }
````

Firestore wants:

```json
{
  "fields": {
    "playerId": { "stringValue": "P01" }
  }
}
````


For a flat document with a few strings, that's manageable. But your save data has deeply nested objects (challenges, attempts, player movement arrays) — converting all of that to Firestore's typed format would be painful and fragile.

The practical solution: store a few metadata fields that Firestore can query (sessionId, playerId, timestamps, deviceName), and store the full game data as a single JSON string. So the Firestore document looks like:

sessionId:   "2026-08-03_14h30m25_P01"     ← queryable
playerId:    "P01"                          ← queryable
startTime:   "2026-08-03T14:30:25Z"         ← queryable
deviceName:  "TabletLabo03"                 ← queryable
gameData:    "{...entire JSON as string...}" ← the payload

This means in Streamlit you'll query Firestore by player or date using the metadata fields, and when you load a session, you parse the gameData string back into a dict — which your app already knows how to do.





# FireStoreUploader

PATCH instead of POST — POST auto-generates a random document ID. PATCH lets us set the document ID to the sessionId, so game_saves/2026-08-03_14h30m25_P01 is the document path. This also means if the same file gets uploaded twice (say the app crashed after upload but before moving the file), it just overwrites with the same data instead of creating a duplicate.

BuildFirestoreDocument — this is where the Firestore format quirk gets handled. It reads the save file back, extracts the metadata fields for Firestore to index, and wraps the entire original JSON into the gameData string field. The Escape() method ensures quotes and newlines inside your game JSON don't break the outer Firestore JSON.

UploadAllUnsent() is public — so your SaveSystem can call it right after saving:

## Escape
The problem is that your game JSON is being placed inside another JSON string. Here's what happens without escaping.

Say your raw game JSON looks like this:

json
{"sessionId": "2026-08-03_14h30m25_P01", "playerId": "P01"}

When we put that into the Firestore document's gameData field, we're writing:

json
"gameData": {"stringValue": "{"sessionId": "2026-08-03_14h30m25_P01"}"}

The parser sees the " before sessionId and thinks the string ended there. The whole document breaks.

With escaping, those inner quotes become \":

json
"gameData": {"stringValue": "{\"sessionId\": \"2026-08-03_14h30m25_P01\"}"}

Now the parser knows those quotes are part of the string content, not string boundaries.

Same logic for the other characters:

\\ — if your data contains a backslash, it needs doubling so it's not mistaken for an escape sequence
\n and \r — literal newlines inside a JSON string are illegal, they need to become the text \n

The true in JsonUtility.ToJson(data, true) adds newlines for readability, which is exactly why the \n escaping matters — without it, the pretty-printed game JSON would break the Firestore wrapper.

In [ ]:
# Fire store page

"""
Firestore connection and data retrieval.
Pure Python — no Streamlit imports.
"""

import json
import firebase_admin
from firebase_admin import credentials, firestore


def get_firestore_client(creds_path: str):
    """
    Initialize Firebase and return a Firestore client.
    Safe to call multiple times — only initializes once.
    """
    if not firebase_admin._apps:
        cred = credentials.Certificate(creds_path)
        firebase_admin.initialize_app(cred)

    return firestore.client()


def get_all_sessions(db) -> list[dict]:
    """
    Fetch all documents from game_saves collection.
    Returns a list of metadata dicts (without gameData).
    """
    docs = db.collection("game_saves").stream()

    sessions = []
    for doc in docs:
        data = doc.to_dict()
        sessions.append({
            "sessionId": data.get("sessionId", ""),
            "playerId": data.get("playerId", ""),
            "startTime": data.get("startTime", ""),
            "saveTime": data.get("saveTime", ""),
            "deviceName": data.get("deviceName", ""),
        })

    return sessions


def get_session_game_data(db, session_id: str) -> dict:
    """
    Fetch a single session's gameData and parse it from JSON string to dict.
    """
    doc = db.collection("game_saves").document(session_id).get()

    if not doc.exists:
        return {}

    data = doc.to_dict()
    game_data_str = data.get("gameData", "{}")
    return json.loads(game_data_str)




## Slit get_game_data_dict

Right now get_game_data_dict does two things in one function:

Opens a file and reads JSON into a Python dict
Parses that dict into dataframes and metadata

When loading from Firestore, step 1 is already done — get_session_game_data gives you the dict directly. But you still need step 2. So instead of duplicating all the parsing logic, we split the existing function in two: